# Eksperimen 15: Cross-Station Spatial Influence
**Notebook 12 | Target: submission_12.csv**

Fondasi terkuat yang pernah kita bangun, dengan penambahan dua lapis inovasi baru:

1. **Cross-Station Spatial Influence Features (baru):** 30 stasiun pengukur di DAS Bengawan Solo rata-rata hanya berjarak 10.6 km satu sama lain. Secara hidrologi, jika satu sungai banjir, sungai tetangga akan mengikuti dalam hitungan jam hingga hari. Untuk setiap stasiun, kita hitung rata-rata TMA tertimbang dari semua stasiun lain (bobot = exp(-jarak_km/30)), memberi model informasi "kondisi lingkungan hidrologi" di cutoff dan mundur 24, 72, 168, 336 jam.

2. **Extended Rolling Windows (14 hari & 28 hari):** Selain 24h, 3d, 7d (dari Exp 14), kita tambahkan akumulasi hujan 14 hari dan 28 hari. Ini menangkap sinyal jenuhnya DAS di musim hujan yang berlangsung berminggu-minggu.

3. **Memanfaatkan Penuh Cuaca Masa Depan:** `data_lingkungan.csv` terverifikasi 100% terisi hingga Mei 2026. Pre-merge rolling windows kita secara otomatis menggunakan data real ini.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import TimeSeriesSplit
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Pemuatan Data

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords   = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime']  = pd.to_datetime(test['id'].str[:19])
test['nama_pos']  = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])
overall_cutoff = train['datetime'].max()
print('Batas train:', overall_cutoff)
print('Test range:', test['datetime'].min(), 'to', test['datetime'].max())

## 2. Profil Stasiun & Anchor TMA (7 Titik Waktu)

In [ ]:
global_mean = train['tma_mdpl'].mean()
global_std  = train['tma_mdpl'].std()

station_profile = train.groupby('nama_pos')['tma_mdpl'].agg(
    tma_mean='mean', tma_std='std',
    tma_p25=lambda x: x.quantile(0.25),
    tma_p75=lambda x: x.quantile(0.75)
).reset_index()
station_profile['tma_std'] = station_profile['tma_std'].fillna(1.0)

train_sorted = train.sort_values(['nama_pos', 'datetime'])
anchor_offsets = {'0h':'0H','24h':'24H','72h':'72H','168h':'168H','336h':'336H','720h':'720H','1440h':'1440H'}

def get_tma_at(pos_df, target_dt):
    subset = pos_df[pos_df['datetime'] <= target_dt]
    return np.nan if subset.empty else subset.iloc[-1]['tma_mdpl']

def get_window_stats(pos_df, end_dt, days):
    start_dt = end_dt - pd.Timedelta(f'{days}D')
    subset = pos_df[(pos_df['datetime'] >= start_dt) & (pos_df['datetime'] <= end_dt)]['tma_mdpl']
    if len(subset) < 2: return np.nan, np.nan
    return subset.std(), np.polyfit(np.arange(len(subset)), subset.values, 1)[0]

anchor_records = []
for pos in train['nama_pos'].unique():
    pos_df = train_sorted[train_sorted['nama_pos'] == pos]
    row = {'nama_pos': pos}
    for label, offset in anchor_offsets.items():
        row[f'tma_anchor_{label}'] = get_tma_at(pos_df, overall_cutoff - pd.Timedelta(offset))
    vol7, trend7  = get_window_stats(pos_df, overall_cutoff, 7)
    vol30, trend30= get_window_stats(pos_df, overall_cutoff, 30)
    row.update({'tma_vol_7d':vol7,'tma_trend_7d':trend7,'tma_vol_30d':vol30,'tma_trend_30d':trend30})
    anchor_records.append(row)

anchor_df = pd.DataFrame(anchor_records)
station_profile = pd.merge(station_profile, anchor_df, on='nama_pos', how='left')
print(station_profile[['nama_pos','tma_anchor_0h','tma_anchor_168h','tma_trend_7d']].to_string())

## 3. Seasonal Historical TMA

In [ ]:
train['month'] = train['datetime'].dt.month
seasonal_mean = train.groupby(['nama_pos','month'])['tma_mdpl'].mean().reset_index()
seasonal_mean.columns = ['nama_pos','month','tma_seasonal_hist_mean']
seasonal_std  = train.groupby(['nama_pos','month'])['tma_mdpl'].std().reset_index()
seasonal_std.columns  = ['nama_pos','month','tma_seasonal_hist_std']
seasonal_profile = pd.merge(seasonal_mean, seasonal_std, on=['nama_pos','month'])
print(f'Profil musiman: {seasonal_profile.shape[0]} baris')

## 4. Cross-Station Spatial Influence Features ⭐ FITUR BARU

**Konsep:**
Untuk setiap stasiun i, kita hitung rata-rata TMA dari semua stasiun tetangga j, dengan bobot:
$$w_{ij} = e^{-d_{ij}/30\ km}$$

Ini mengkodekan pengetahuan hidrologi: sungai yang berdekatan cenderung co-move. Jika tetangga semua sedang tinggi, prediksi juga akan tinggi.

In [ ]:
sp_coords = pd.merge(station_profile[['nama_pos']], coords, on='nama_pos', how='left')
lats = sp_coords['latitude'].values
lons = sp_coords['longitude'].values
stations_list = sp_coords['nama_pos'].tolist()
n_st = len(stations_list)

def haversine_matrix(lats, lons):
    R = 6371.0
    lat_r = np.radians(lats[:,None] - lats[None,:])
    lon_r = np.radians(lons[:,None] - lons[None,:])
    a = (np.sin(lat_r/2)**2
         + np.cos(np.radians(lats[:,None]))*np.cos(np.radians(lats[None,:]))*np.sin(lon_r/2)**2)
    return 2*R*np.arcsin(np.sqrt(np.clip(a,0,1)))

D = haversine_matrix(lats, lons)
np.fill_diagonal(D, np.inf)
SCALE_KM = 30.0

print('Jarak antar stasiun (km):')
print(f'  Rata-rata nearest neighbor: {D.min(axis=1).mean():.1f} km')
print(f'  Max nearest neighbor: {D.min(axis=1).max():.1f} km')

for label in ['0h','24h','72h','168h','336h']:
    anchor_vals = station_profile[f'tma_anchor_{label}'].values.astype(float)
    wmeans, mxnn3 = [], []
    for i in range(n_st):
        weights = np.exp(-D[i] / SCALE_KM)
        valid   = ~np.isnan(anchor_vals)
        w = weights * valid
        ws = w.sum()
        wmeans.append(np.nansum(w * np.nan_to_num(anchor_vals)) / ws if ws > 0 else np.nan)
        nn3 = np.argsort(D[i])[:3]
        mxnn3.append(np.nanmax(anchor_vals[nn3]) if np.any(valid[nn3]) else np.nan)
    station_profile[f'spatial_tma_{label}']  = wmeans
    station_profile[f'max_nn3_tma_{label}']  = mxnn3

anchor_0h = station_profile['tma_anchor_0h'].values.astype(float)
nn_std, nn_range = [], []
for i in range(n_st):
    nn5  = np.argsort(D[i])[:5]
    vals = anchor_0h[nn5]
    nn_std.append(np.nanstd(vals))
    nn_range.append(np.nanmax(vals)-np.nanmin(vals) if np.any(~np.isnan(vals)) else np.nan)
station_profile['neighbor_tma_std_0h']   = nn_std
station_profile['neighbor_tma_range_0h'] = nn_range

print('Spatial features preview:')
print(station_profile[['nama_pos','tma_anchor_0h','spatial_tma_0h','max_nn3_tma_0h']].head(8).to_string())

## 5. Pre-Merge 6H Rolling (Fix Data Loss Bug + Extended Windows)

In [ ]:
env_data = env_data.sort_values(['nama_pos','datetime'])
macro_cols   = ['nino_34','mjo_phase','mjo_amplitude','mjo_active','rmm1','rmm2']
dynamic_cols = ['surface_pressure_hpa','pressure_msl_hpa','soil_moisture_0_7cm',
                'soil_moisture_7_28cm','soil_moisture_28_100cm','soil_moisture_100_255cm']
for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()
for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(
        lambda x: x.interpolate(method='linear').bfill().ffill()
    ).reset_index(level=0, drop=True)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude','longitude']])

def aggregate_env_6h_rolling(df):
    agg_funcs = {col: 'mean' for col in df.columns if col not in ['nama_pos','landcover_name','datetime']}
    agg_funcs['rainfall_mm']='sum'; agg_funcs['rainfall_openmeteo_mm']='sum'; agg_funcs['rainfall_max_24h_mm']='max'
    idx = df.set_index('datetime')
    agg = idx.groupby(['nama_pos', pd.Grouper(freq='6h', label='right', closed='right')]).agg(agg_funcs).reset_index()
    for w, label in [(4,'24h'),(12,'3d'),(28,'7d'),(56,'14d'),(112,'28d')]:
        agg[f'rainfall_roll_{label}']       = agg.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(w, min_periods=1).sum())
        agg[f'soil_roll_{label}']           = agg.groupby('nama_pos')['soil_moisture_0_7cm'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        agg[f'pressure_roll_{label}']       = agg.groupby('nama_pos')['surface_pressure_hpa'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        agg[f'temp_roll_{label}']           = agg.groupby('nama_pos')['temperature_c'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        agg[f'rain_openmeteo_roll_{label}'] = agg.groupby('nama_pos')['rainfall_openmeteo_mm'].transform(lambda x: x.rolling(w, min_periods=1).sum())
    return agg

env_agg = aggregate_env_6h_rolling(env_data)
env_cutoff = env_agg[env_agg['datetime'] <= overall_cutoff].copy()
env_cutoff_last = env_cutoff.sort_values('datetime').groupby('nama_pos').last().reset_index()
env_cutoff_last = env_cutoff_last[['nama_pos','rainfall_mm','soil_moisture_0_7cm','surface_pressure_hpa','temperature_c']].rename(
    columns={'rainfall_mm':'rain_anchor_0h','soil_moisture_0_7cm':'soil_anchor_0h','surface_pressure_hpa':'pressure_anchor_0h','temperature_c':'temp_anchor_0h'})
env_7d = env_cutoff[env_cutoff['datetime'] >= overall_cutoff-pd.Timedelta('7D')].groupby('nama_pos').agg(
    rain_sum_7d=('rainfall_mm','sum'), soil_mean_7d=('soil_moisture_0_7cm','mean')).reset_index()
station_profile = pd.merge(station_profile, env_cutoff_last, on='nama_pos', how='left')
station_profile = pd.merge(station_profile, env_7d, on='nama_pos', how='left')
print(f'env_agg shape: {env_agg.shape}')

## 6. Merge & Feature Engineering

In [ ]:
test['tma_mdpl'] = np.nan
all_data = pd.concat([train, test], ignore_index=True).sort_values(['nama_pos','datetime']).reset_index(drop=True)
all_data = pd.merge(all_data, env_agg, on=['datetime','nama_pos'], how='left')
all_data = pd.merge(all_data, coords,  on='nama_pos', how='left')
all_data = pd.merge(all_data, station_profile, on='nama_pos', how='left')
all_data['tma_mean'] = all_data['tma_mean'].fillna(global_mean)
all_data['tma_std']  = all_data['tma_std'].fillna(global_std)

all_data['month']      = all_data['datetime'].dt.month
all_data['hour']       = all_data['datetime'].dt.hour
all_data['day_of_year']= all_data['datetime'].dt.dayofyear
all_data['sin_hour']   = np.sin(2*np.pi*all_data['hour']/24)
all_data['cos_hour']   = np.cos(2*np.pi*all_data['hour']/24)
all_data['sin_month']  = np.sin(2*np.pi*all_data['month']/12)
all_data['cos_month']  = np.cos(2*np.pi*all_data['month']/12)
all_data['sin_doy']    = np.sin(2*np.pi*all_data['day_of_year']/365)
all_data['cos_doy']    = np.cos(2*np.pi*all_data['day_of_year']/365)
le = LabelEncoder(); all_data['nama_pos_encoded'] = le.fit_transform(all_data['nama_pos'])
all_data['steps_ahead'] = (all_data['datetime'] - overall_cutoff) / pd.Timedelta('3H')
all_data['hours_ahead'] = all_data['steps_ahead'] * 3
all_data['days_ahead']  = all_data['steps_ahead'] / 8

for label in anchor_offsets:
    col = f'tma_anchor_{label}'
    all_data[col+'_norm'] = (all_data[col] - all_data['tma_mean']) / all_data['tma_std']

steps_clip = np.clip(all_data['steps_ahead'], 0, None)
all_data['decay_short']  = all_data['tma_anchor_0h_norm'] * np.exp(-steps_clip/28)
all_data['decay_mid']    = all_data['tma_anchor_0h_norm'] * np.exp(-steps_clip/56)
all_data['decay_long']   = all_data['tma_anchor_0h_norm'] * np.exp(-steps_clip/168)
all_data['decay_vlong']  = all_data['tma_anchor_0h_norm'] * np.exp(-steps_clip/336)
all_data['trend_signal'] = all_data['tma_trend_7d']  * np.exp(-steps_clip/112)
all_data['trend_long']   = all_data['tma_trend_30d'] * np.exp(-steps_clip/336)

for label in ['0h','24h','72h','168h']:
    scol = f'spatial_tma_{label}'
    all_data[scol+'_norm'] = (all_data[scol] - all_data['tma_mean']) / all_data['tma_std']
all_data['spatial_decay_short'] = all_data['spatial_tma_0h_norm'] * np.exp(-steps_clip/56)
all_data['spatial_decay_long']  = all_data['spatial_tma_0h_norm'] * np.exp(-steps_clip/168)

all_data = pd.merge(all_data, seasonal_profile, on=['nama_pos','month'], how='left')
all_data['seasonal_norm']      = (all_data['tma_seasonal_hist_mean'] - all_data['tma_mean']) / all_data['tma_std']
all_data['seasonal_deviation'] = all_data['tma_anchor_0h_norm'] - all_data['seasonal_norm']
all_data['runoff_factor']      = all_data['rainfall_mm'] * all_data['soil_moisture_0_7cm']
all_data['pressure_drop']      = all_data.groupby('nama_pos')['surface_pressure_hpa'].diff(1).fillna(0)
all_data['temp_humidity_index']= all_data['temperature_c'] * all_data['humidity_pct'] / 100
all_data['cloud_rain_index']   = all_data['cloud_cover_pct'] * all_data['rainfall_mm']
print(f'all_data shape: {all_data.shape}')

## 7. Training & Submisi

In [ ]:
train_mask = all_data['tma_mdpl'].notnull()
all_data['tma_normalized'] = np.nan
all_data.loc[train_mask,'tma_normalized'] = (
    (all_data.loc[train_mask,'tma_mdpl'] - all_data.loc[train_mask,'tma_mean'])
    / all_data.loc[train_mask,'tma_std']
)
train_data = all_data[train_mask].sort_values('datetime').reset_index(drop=True)
test_data  = all_data[~train_mask].sort_values('datetime').reset_index(drop=True)

anchor_raw_cols  = [f'tma_anchor_{l}' for l in anchor_offsets]
spatial_raw_cols = [f'spatial_tma_{l}' for l in ['0h','24h','72h','168h','336h']]
drop_cols = ['datetime','nama_pos','tma_mdpl','tma_normalized','id','landcover_name'] + anchor_raw_cols + spatial_raw_cols
features  = [c for c in train_data.columns if c not in drop_cols]
X_full, y_full = train_data[features], train_data['tma_normalized']
X_test = test_data[features]
tscv   = TimeSeriesSplit(n_splits=5)
print(f'Total fitur: {len(features)}')
print(features)

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators':5000,
        'learning_rate':trial.suggest_float('learning_rate',0.005,0.05,log=True),
        'num_leaves':trial.suggest_int('num_leaves',31,255),
        'max_depth':trial.suggest_int('max_depth',5,12),
        'subsample':trial.suggest_float('subsample',0.5,0.9),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.5,0.9),
        'min_child_samples':trial.suggest_int('min_child_samples',30,300),
        'reg_alpha':trial.suggest_float('reg_alpha',0.1,20.0,log=True),
        'reg_lambda':trial.suggest_float('reg_lambda',0.1,20.0,log=True),
        'subsample_freq':1,'random_state':42,'verbose':-1
    }
    scores = []
    for tr_idx,va_idx in tscv.split(X_full):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_full.iloc[tr_idx], y_full.iloc[tr_idx],
              eval_set=[(X_full.iloc[va_idx], y_full.iloc[va_idx])],
              callbacks=[lgb.early_stopping(300, verbose=False)])
        scores.append(mean_squared_error(y_full.iloc[va_idx], m.predict(X_full.iloc[va_idx])))
    return np.mean(scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective_lgb, n_trials=60)
best_lgb = study.best_params
best_lgb.update({'n_estimators':5000,'random_state':42,'verbose':-1,'subsample_freq':1})
print('Best params:', best_lgb)

In [ ]:
test_preds = np.zeros(len(X_test))
cv_scores  = []
print('K-Fold Final Training...')
for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_full)):
    X_tr,X_va = X_full.iloc[tr_idx],X_full.iloc[va_idx]
    y_tr,y_va = y_full.iloc[tr_idx],y_full.iloc[va_idx]
    val_mean = train_data.iloc[va_idx]['tma_mean'].values
    val_std  = train_data.iloc[va_idx]['tma_std'].values
    m = lgb.LGBMRegressor(**best_lgb)
    m.fit(X_tr,y_tr,eval_set=[(X_va,y_va)],callbacks=[lgb.early_stopping(300,verbose=False)])
    p_abs = (m.predict(X_va)*val_std)+val_mean
    y_abs = (y_va.values*val_std)+val_mean
    fold_rmse = np.sqrt(mean_squared_error(y_abs,p_abs))
    cv_scores.append(fold_rmse)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.4f} | Trees: {m.best_iteration_}')
    test_preds += m.predict(X_test)/tscv.n_splits

print(f'\nRata-rata CV RMSE (Exp15 Spatial): {np.mean(cv_scores):.4f}')

test_mean = test_data['tma_mean'].values; test_std = test_data['tma_std'].values
final_tma = (test_preds*test_std)+test_mean
os.makedirs('../submissions', exist_ok=True)
pd.DataFrame({'id':test_data['id'],'tma_mdpl':final_tma}).to_csv('../submissions/submission_12.csv',index=False)
print('Selesai. submission_12.csv tersimpan.')